In [29]:
from typing import Tuple, List, Optional
State = Tuple[int, ...]

def successors(state: State) -> List[State]:
  #generate all possible successor states from the given state. for each heap, try removing 1..heap_size stones and return all resulting states in order (heap index ascending)
    moves = []
    for i, stones in enumerate(state):
        if stones == 0:
            continue #remove more stones first (descending order)
        for take in range(stones, 0, -1):
            new_state = list(state)
            new_state[i] -= take
            moves.append(tuple(new_state))
    return moves

def is_terminal(state: State) -> bool:
  #a state is terminal if all heaps are empty
    return all(stones == 0 for stones in state)


def utility(state: State, maximizing_turn: bool) -> int:
  #if max turn, then min took last stone -> min wins -> utility = -1, if min turn, then max took the last stone -> max wins -> utility = +1
    if not is_terminal(state):
        raise ValueError("utility() should be called iff terminal states")
    return -1 if maximizing_turn else +1

In [30]:
class Counters:
    def __init__(self):
        self.nodes_explored = 0
        self.nodes_pruned = 0


class MinimaxCounters:
    def __init__(self):
        self.nodes_explored = 0

def alpha_beta(state: State, alpha: int, beta: int, maximizing_turn: bool,
               counters: Counters, verbose: bool = True) -> int:
  #returns the minimax value (+1 if MAX can force a win, -1 otherwise).
    counters.nodes_explored += 1
    player = "MAX" if maximizing_turn else "MIN"

    if verbose:
        print(f"{player} explores {state}")

    if is_terminal(state):
        val = utility(state, maximizing_turn)
        if verbose:
            print(f"terminal reached at {state}, utility = {val}")
        return val

    best_value = float('-inf') if maximizing_turn else float('inf')
    succs = successors(state)

    for i, s in enumerate(succs):
        if verbose:
            print(f"Considering move: {state} -> {s}")
        val = alpha_beta(s, alpha, beta, not maximizing_turn, counters, verbose)

        if maximizing_turn:
            best_value = max(best_value, val)
            alpha = max(alpha, best_value)
        else:
            best_value = min(best_value, val)
            beta = min(beta, best_value)

        #prune if possible
        if alpha >= beta:
            skipped = len(succs) - (i + 1)  #how many moves we skip
            counters.nodes_pruned += skipped
            if verbose:
                print(f"[pruned {skipped} successor(s) at {state} "
                      f"because alpha >= beta (alpha={alpha}, beta={beta})]")
            break

    return int(best_value)

def minimax(state: State, maximizing_turn: bool, counters: MinimaxCounters) -> int:
    counters.nodes_explored += 1

    if is_terminal(state):
        return utility(state, maximizing_turn)

    if maximizing_turn:
        return max(minimax(s, False, counters) for s in successors(state))
    else:
        return min(minimax(s, True, counters) for s in successors(state))

In [31]:
def find_best_move_alpha_beta(initial_state: State, verbose: bool = True):
  #run alpha beta from the initial state and return the best move for max
    counters = Counters()
    alpha, beta = float('-inf'), float('inf')
    best_value = float('-inf')
    best_move: Optional[State] = None

    if verbose:
        print(f"Initial State: {initial_state}")
        print("Starting Alpha–Beta search...\n")

    for s in successors(initial_state):
        if verbose:
            print(f"Considering move: {initial_state} -> {s}")
        val = alpha_beta(s, alpha, beta, False, counters, verbose)
        if verbose:
            print(f"Value for move {initial_state} -> {s} is {val}\n")

        if val > best_value:
            best_value = val
            best_move = s
        alpha = max(alpha, best_value)

    if verbose:
        print("search done\n")
        print(f"best move for max: {initial_state} -> {best_move}")
        print(f"outcome: {'winner' if best_value == 1 else 'loser'}")
        print(f"nodes explored: {counters.nodes_explored}")
        print(f"nodes pruned: {counters.nodes_pruned}")

    return best_move, best_value, counters

In [32]:
def compare_with_minimax(initial_state: State):
  #plain minimax for comparison (counts nodes explored)
    counters = MinimaxCounters()
    val = minimax(initial_state, True, counters)
    print("\plain minmax")
    print(f"value from initial state: {val}")
    print(f"nodes explored: {counters.nodes_explored}")
    return val, counters


if __name__ == "__main__":
    #user input
    raw = input("initial nim state (comma-separated, e.g: 3,4,5) ")
    try:
        #input string into a tuple of ints
        initial = tuple(int(x.strip()) for x in raw.split(","))
    except ValueError:
        print("invalid input. enter integers separated by commas (e.g: 3,4,5)")
        exit(1)

    #alpha beta
    best_move, best_value, ab_counters = find_best_move_alpha_beta(initial, verbose=True)

    #plain minimax
    mm_value, mm_counters = compare_with_minimax(initial)

    #summary
    print("\nsummary")
    print(f"initial state: {initial}")
    print(f"best move: {initial} -> {best_move}")
    print(f"alpha beta val: {best_value}")
    print(f"nodes explored: {ab_counters.nodes_explored}")
    print(f"nodes pruned: {ab_counters.nodes_pruned}")
    print(f"nodes explored (minmax): {mm_counters.nodes_explored}")

<>:5: SyntaxWarning: invalid escape sequence '\p'
<>:5: SyntaxWarning: invalid escape sequence '\p'
/tmp/ipython-input-1722221951.py:5: SyntaxWarning: invalid escape sequence '\p'
  print("\plain minmax")


Streaming output truncated to the last 5000 lines.
Considering move: (0, 1, 1) -> (0, 1, 0)
MIN explores (0, 1, 0)
Considering move: (0, 1, 0) -> (0, 0, 0)
MAX explores (0, 0, 0)
terminal reached at (0, 0, 0), utility = -1
[pruned 0 successor(s) at (0, 1, 0) because alpha >= beta (alpha=-1, beta=-1)]
[pruned 3 successor(s) at (0, 1, 5) because alpha >= beta (alpha=-1, beta=-1)]
Considering move: (1, 1, 5) -> (1, 0, 5)
MIN explores (1, 0, 5)
Considering move: (1, 0, 5) -> (0, 0, 5)
MAX explores (0, 0, 5)
Considering move: (0, 0, 5) -> (0, 0, 0)
MIN explores (0, 0, 0)
terminal reached at (0, 0, 0), utility = 1
[pruned 4 successor(s) at (0, 0, 5) because alpha >= beta (alpha=1, beta=1)]
Considering move: (1, 0, 5) -> (1, 0, 0)
MAX explores (1, 0, 0)
Considering move: (1, 0, 0) -> (0, 0, 0)
MIN explores (0, 0, 0)
terminal reached at (0, 0, 0), utility = 1
[pruned 0 successor(s) at (1, 0, 0) because alpha >= beta (alpha=1, beta=1)]
Considering move: (1, 0, 5) -> (1, 0, 1)
MAX explores (1, 0